# Build your own HALT comparison

Load saved measurements and ask your own questions. This notebook never runs a model.
The first analyses need only HALT and the Python standard library. Optional cells use
pandas and matplotlib, if installed in this notebook's Python environment.

Edit the path and baseline below. A run directory, `results.csv`, `results.jsonl`, or
`per_example.jsonl` can be loaded. Paths are relative to the kernel's working directory,
which the first cell prints; an absolute path also works. Keep failed, incomplete, and
abstained runs in the data so accuracy retains its original denominator.

In [ ]:
from pathlib import Path

from halt.evaluation import compare, load_results, render_comparison

RUN_DIRECTORY = Path("../runs/my_trial")  # Edit this path to your saved experiment.
BASELINE = "full_reasoning"

print("Kernel working directory:", Path.cwd())
rows = load_results(RUN_DIRECTORY)
summary = compare(rows, baseline=BASELINE, seed=0)
print(summary["interpretation"])
print(render_comparison(summary))


## Check the experiment and coverage

`compare` checks recorded dataset, model, settings, code identity, method configurations,
and duplicate observations before pairing question IDs and seeds. Its checks do not
establish independent questions or identical machine load. All rows contribute to each
method's overall accuracy; paired effects use only matched question/seed observations.
Confidence intervals are fractions, while the terminal display uses percentage points.
The API suppresses an independent-item interval for adaptive sessions or repeated questions.

In [ ]:
print("Experiment:", summary["experiment"])
print("Seeds:", summary["seeds"])
for method in summary["methods"]:
    print(
        method["method_id"],
        "rows:", method["sample_count"],
        "paired:", method["paired_sample_count"],
        "unpaired:", method["unpaired_sample_count"],
        "baseline unmatched:", method["baseline_unmatched_sample_count"],
        "statuses:", method["status_counts"],
    )
    print("  Uncertainty:", method["uncertainty_note"])


## Find answers that changed from correct to wrong

This is a diagnostic slice, not a replacement accuracy estimate. It retains non-completed
outcomes when a baseline answer was correct. Join your own question categories using
`question_id`; retain the matching baseline and seed when comparing a subset.

In [ ]:
baseline_rows = {
    (row["question_id"], row["seed"]): row
    for row in rows if row["method_id"] == BASELINE
}
regressions = []
for row in rows:
    baseline_row = baseline_rows.get((row["question_id"], row["seed"]))
    if row["method_id"] != BASELINE and baseline_row and baseline_row["correct"] and not row["correct"]:
        regressions.append({key: row[key] for key in (
            "question_id", "seed", "method_id", "question", "reference", "answer",
            "status", "stop_reason", "total_generated_tokens", "elapsed_seconds",
        )})
print("Correct-to-wrong matched observations:", len(regressions))
for regression in regressions[:10]:
    print(regression)


## Optional pandas table and export

If needed, install `pandas` in the notebook's Python environment, then rerun this cell.
No rows are discarded here. Accuracy is shown as a percentage; other columns keep their
recorded units. Generated tokens include reasoning, final answers, and **all generated
probe output**. Processed input tokens include repeated processing. Do not add processed
input, recomputed-prefix, and scored-token counts together: their scopes overlap.

Set `SAVE_CUSTOM_CSV` to `True` to save this custom summary beside your original results.
It writes `custom_accuracy_work.csv`; the original HALT exports remain available.

In [ ]:
import importlib.util

SAVE_CUSTOM_CSV = False
frame = None
comparison_frame = None
if importlib.util.find_spec("pandas") is None:
    print("Optional table skipped: install pandas to use this cell.")
else:
    import pandas as pd

    frame = pd.DataFrame(rows)
    comparison_frame = pd.DataFrame(summary["methods"])
    comparison_frame["accuracy_percent"] = 100 * comparison_frame["accuracy"]
    table_columns = [
        "method_id", "sample_count", "accuracy_percent", "failure_count",
        "mean_reasoning_tokens", "mean_answer_tokens", "mean_probe_output_tokens",
        "mean_total_generated_tokens", "mean_input_tokens", "mean_probe_calls",
        "mean_latency_seconds", "timing_sample_count", "timing_scope",
    ]
    custom_table = comparison_frame[table_columns]
    display(custom_table)
    if SAVE_CUSTOM_CSV:
        output_directory = RUN_DIRECTORY if RUN_DIRECTORY.is_dir() else RUN_DIRECTORY.parent
        output_path = output_directory / "custom_accuracy_work.csv"
        custom_table.to_csv(output_path, index=False)
        print("Saved", output_path)


## Optional accuracy/work plot

This chart uses the same summary as HALT's table. It shows a generation tradeoff, not a
claim about total compute savings. Synthetic token counts are fragments and must not be
presented as measured model performance. Accuracy includes non-completed rows.

Real-run latency includes probes, controller work, and answer finalization; it excludes
model loading and report generation. Missing timing stays missing. Scripted and replay
measurements do not support inference-speed claims, so the latency plot below is limited
to results labeled `real`. If needed, install `matplotlib` to use this cell.

In [ ]:
if importlib.util.find_spec("matplotlib") is None:
    print("Optional charts skipped: install matplotlib to use this cell.")
else:
    import matplotlib.pyplot as plt

    methods = summary["methods"]
    evidence_kind = summary["experiment"]["evidence_kind"]
    fig, ax = plt.subplots(figsize=(8, 4.5))
    for method in methods:
        x, y = method["mean_total_generated_tokens"], 100 * method["accuracy"]
        ax.scatter(x, y)
        ax.annotate(method["method_id"], (x, y), xytext=(5, 5), textcoords="offset points")
    ax.set_xlabel("Mean generated tokens per observation (reasoning + answer + probes)")
    ax.set_ylabel("Accuracy (%) ? all outcomes")
    ax.set_title(f"Accuracy and generation work | {evidence_kind} | {summary['experiment']['model_id']}")
    fig.tight_layout()
    plt.show()

    if evidence_kind == "real":
        fig, ax = plt.subplots(figsize=(8, 4.5))
        for method in methods:
            seconds = method["mean_latency_seconds"]
            if seconds is None:
                continue
            accuracy = 100 * method["accuracy"]
            ax.scatter(seconds, accuracy)
            ax.annotate(method["method_id"], (seconds, accuracy), xytext=(5, 5), textcoords="offset points")
        ax.set_xlabel("Mean end-to-end inference time per timed observation (seconds)")
        ax.set_ylabel("Accuracy (%) ? all outcomes")
        ax.set_title("Measured run latency; model loading excluded")
        fig.tight_layout()
        plt.show()
    else:
        print("Inference-latency chart unavailable for", evidence_kind, "measurements.")


## Compare a chosen subset or several experiments

For a category comparison, choose question IDs and keep **every method's matching rows**.
Review paired/unpaired counts again; a diagnostic subset changes the population represented
by an accuracy estimate. Do not select only correct/completed runs and call that overall
accuracy. To compare another model or dataset, load its output directory and call
`grouped_compare` so incompatible experiments remain separate.

```python
from halt.evaluation import grouped_compare

chosen_ids = {"q1", "q2"}
subset = [row for row in rows if row["question_id"] in chosen_ids]
subset_summary = compare(subset, baseline=BASELINE)  # Requires a nonempty subset and baseline.

other_rows = load_results("../runs/another_trial")
separate_comparisons = grouped_compare(rows + other_rows, baseline=BASELINE)
```

See [the result schema and measurement definitions](../docs/results_format.md) for field
meanings, missing values, timing scope, failure accounting, and uncertainty limitations.